# LF reversal v2 — UP-anchored, approved-drug-filtered

Strengthened reversal for Arm A, run on the **cross-platform-replicated UP program** (activated / matrix-depositing fibroblast signature; replicated in bulk GSE113212 at rank-p 1.5e-7). Two upgrades over the dry-run that was swamped by cytotoxic tool compounds:

1. **UP-anchored, one-sided** — query for perturbagens that push the replicated fibrotic UP genes *down*. The pilot DOWN pole did not replicate, so it is dropped.
2. **Approved-drug filter + real MoA** — L1000CDS2 hits are joined to the **Broad Drug Repurposing Hub** and filtered to **Launched (approved), non-oncology** drugs, with frankly cytotoxic mechanisms (HDAC / topoisomerase / tubulin / proteasome / CDK / antimetabolite / DNA-damaging) removed. Kinase inhibitors are kept (nintedanib-class anti-fibrotics are legitimate).

> **Still discovery-stage, not confirmatory.** The signature is cross-platform replicated but the single-cell side is one cohort; L1000CDS2 returns a bounded top set; name/MoA joins are imperfect. Output is a *hypothesis shortlist* to triage for local-delivery feasibility and design-around vs Pitt US2024/0026359, not a validated candidate.

In [ ]:
# 1. Install
!pip -q install requests pandas 2>/dev/null
print("done")

In [ ]:
# 2. Imports + load the replicated UP signature
import os, re, json, io
import requests
import pandas as pd

# Replicated UP program (GSE294458 pilot, confirmed in bulk GSE113212). Falls back
# to the 'up' list of out/LF_reversal_signature.json if present in this runtime.
EMBEDDED_UP = ["COL1A2","COL3A1","LUM","ASPN","COL1A1","HTRA1","MFGE8","OGN","TSC22D1",
    "COL5A2","S100A4","ARL6IP5","CD9","DCN","DKK3","CLU","MXRA8","NUPR1","SSPN","SOX5"]
if os.path.exists("out/LF_reversal_signature.json"):
    up_genes = json.load(open("out/LF_reversal_signature.json"))["up"]
    print("using saved full UP signature")
else:
    up_genes = EMBEDDED_UP
    print("using embedded replicated UP top-genes")
print(f"UP genes ({len(up_genes)}):", ", ".join(up_genes[:20]))

## Step 1 — L1000CDS2 reverse query (UP-anchored)

Sends only the UP set (empty down set) so the query is one-sided: find perturbagens whose signature opposes the fibrotic UP program.

In [ ]:
# 3. L1000CDS2 reverse, UP-anchored
URL = "https://maayanlab.cloud/L1000CDS2/query"
def l1000_reverse(up, dn):
    payload = {"data": {"upGenes": up, "dnGenes": dn},
               "config": {"aggravate": False, "searchMethod": "geneSet",
                          "share": True, "combination": False, "db-version": "latest"}}
    r = requests.post(URL, json=payload, headers={"content-type":"application/json"}, timeout=120)
    r.raise_for_status(); return r.json()

res = l1000_reverse(up_genes, [])
if not res.get("topMeta"):
    print("empty with one-sided query; retrying with a small placeholder down set")
    res = l1000_reverse(up_genes, up_genes[-3:])
top = res.get("topMeta", [])
print("returned", len(top), "signatures")

rows = []
for m in top:
    d = (m.get("pert_desc") or "").strip()
    if not d or d in ("-666","unannotated"): continue
    rows.append({"drug": d, "score": m.get("score"),
                 "pubchem_id": m.get("pubchem_id"), "cell_id": m.get("cell_id")})
raw = pd.DataFrame(rows)
agg = (raw.groupby("drug").agg(best_score=("score","min"), n_sigs=("score","size"),
                               pubchem_id=("pubchem_id","first")).reset_index()
          if len(raw) else pd.DataFrame(columns=["drug","best_score","n_sigs","pubchem_id"]))
print("distinct perturbagens:", len(agg))

## Step 2 — Annotate with the Broad Drug Repurposing Hub

Downloads the Hub table (clinical phase, mechanism of action, target, disease area, indication) and joins by salt-stripped name.

In [ ]:
# 4. Repurposing Hub annotation
HUB_URL = "https://s3.amazonaws.com/data.clue.io/repurposing/downloads/repurposing_drugs_20200324.txt"
txt = requests.get(HUB_URL, timeout=120).content.decode("latin-1")
lines = [ln for ln in txt.splitlines() if not ln.startswith("!")]
hub = pd.read_csv(io.StringIO("\n".join(lines)), sep="\t")
hub.columns = [c.strip() for c in hub.columns]
print("hub drugs:", len(hub), "| columns:", list(hub.columns))

SALTS = ["hydrochloride","dihydrochloride","hydrobromide","sulfate","sulphate","mesylate",
    "maleate","citrate","tartrate","phosphate","sodium","potassium","calcium","acetate",
    "fumarate","succinate","hydrate","dihydrate","monohydrate","besylate","tosylate","napsylate"]
def norm(s):
    s = str(s).lower()
    for salt in SALTS: s = s.replace(salt, "")
    return re.sub(r"[^a-z0-9]", "", s)

hub["_k"] = hub["pert_iname"].map(norm)
hub_lut = hub.drop_duplicates("_k").set_index("_k")
agg["_k"] = agg["drug"].map(norm)
for col in ["clinical_phase","moa","target","disease_area","indication"]:
    agg[col] = agg["_k"].map(hub_lut[col])
matched = agg["clinical_phase"].notna().sum()
print(f"annotated {matched}/{len(agg)} perturbagens via the Hub")

## Step 3 — Filter to approved, non-oncology, non-cytotoxic; rank

Tier A = Launched (approved) drugs, disease area not oncology, mechanism not frankly cytotoxic. Tier B = the same but clinical-stage (Phase 1–3).

In [ ]:
# 5. Filter + guards + tiers
CYTO_MOA = ["hdac inhibitor","histone deacetylase","topoisomerase","tubulin","microtubule",
    "proteasome","cdk inhibitor","cyclin-dependent kinase","aurora kinase","polo-like","plk1",
    "protein synthesis inhibitor","dna synthesis","dna alkylating","alkylating agent",
    "ribonucleotide reductase","hsp90","heat shock protein 90","kinesin","survivin",
    "dihydrofolate reductase","dna methyltransferase","parp inhibitor","antineoplastic"]
def is_cytotox(moa): return any(k in str(moa).lower() for k in CYTO_MOA)
def is_onc(da): return "oncology" in str(da).lower()

NAMED_FOR_LF = {"sirolimus","rapamycin","everolimus","temsirolimus","rolipram","cyclopamine",
    "nacetylcysteine","acetylcysteine","2deoxydglucose","2deoxyglucose","decorin"}
CLEAN_FOR_LF = {"pirfenidone","nintedanib","metformin","dasatinib","quercetin","navitoclax",
    "fisetin","simvastatin","atorvastatin","lovastatin","pravastatin","rosuvastatin","fluvastatin"}

a = agg.copy()
a["cytotox_moa"] = a["moa"].map(is_cytotox)
a["oncology"] = a["disease_area"].map(is_onc)
a["novelty"] = "novel-for-LF"
a.loc[a["_k"].isin(NAMED_FOR_LF), "novelty"] = "named-for-LF"
a.loc[a["_k"].isin(CLEAN_FOR_LF), "novelty"] = "clean-for-LF (flag)"

annot = a[a["clinical_phase"].notna()]
keep = annot[~annot["cytotox_moa"] & ~annot["oncology"]]
tierA = keep[keep["clinical_phase"]=="Launched"].sort_values("best_score")
tierB = keep[keep["clinical_phase"].isin(["Phase 3","Phase 2","Phase 1","Phase 2/Phase 3","Phase 1/Phase 2"])].sort_values("best_score")

cols = ["drug","best_score","n_sigs","clinical_phase","moa","disease_area","indication","novelty"]
os.makedirs("out", exist_ok=True)
keep[cols + ["cytotox_moa","oncology","target"]].to_csv("out/LF_reversal_v2_candidates.csv", index=False)

print(f"annotated={len(annot)}  removed: oncology={int(annot['oncology'].sum())}, "
      f"cytotoxic-MoA={int(annot['cytotox_moa'].sum())}  ->  kept={len(keep)}")
print("\n=== TIER A: APPROVED (Launched), non-oncology, non-cytotoxic ===")
print(tierA[cols].to_string(index=False) if len(tierA) else "  (none in this query's top set)")
print("\n=== TIER B: clinical-stage (Phase 1-3), non-oncology, non-cytotoxic ===")
print(tierB[cols].to_string(index=False) if len(tierB) else "  (none)")
print("\nSaved out/LF_reversal_v2_candidates.csv")
print("\nUnannotated / not-in-Hub perturbagens (context):",
      ", ".join(sorted(a[a['clinical_phase'].isna()]['drug'])[:20]))

## Interpretation

- **Tier A is the shortlist that matters**: approved, non-oncology, non-cytotoxic drugs predicted to reverse a *cross-platform-replicated* fibrotic signature. Each is a repurposing hypothesis to triage for (a) plausibility as an anti-fibrotic, (b) **local intra-ligamentous / epidural delivery** feasibility, and (c) novelty vs the LF prior-art memo and design-around vs Pitt US2024/0026359 (miRNA, so a small molecule already differentiates).
- **A `clean-for-LF` flag in Tier A** (pirfenidone, nintedanib, a statin, a senolytic) would be the strongest outcome — a drug with no LF-specific prior art, an anti-fibrotic rationale, and now a data-driven reversal signal on a replicated signature.
- **If Tier A is thin or empty**, that is itself informative: L1000CDS2 returns a bounded top set dominated by tool compounds, so few approved drugs may appear. The fix is a full-library scoring engine (iLINCS / SigCom LINCS, or CMap scored across the whole Repurposing Hub) rather than L1000CDS2's top-50 — a larger build worth doing only if this shortlist looks promising.
- **Nothing here is a candidate for filing.** It is a ranked hypothesis set on a discovery-stage, single-scRNA-cohort (but platform-replicated) signature. Wet-lab validation and FTO opinion remain external. Not medical or legal advice.